# Notebook 08 — Objectif 2 : Environnement × comportement (Summer 2019)

**Objectif SOW :** analyser la relation entre les conditions environnementales et le comportement locomoteur.

**Tâche 2.1 — Synchronisation :** fusionner les données environnementales (HOBO : température, humidité)
avec l'activité accélérométrique (IceTag) sur une base temporelle commune (bins de 15 min).

**Tâche 2.2 — Analyse exploratoire :** étudier comment l'activité locomotrice varie avec la température
et l'indice de stress thermique (THI).

**Pourquoi Summer 2019 :** 18 vaches avec IceTag par vache, 34 fichiers HOBO, et surtout une vraie
variation thermique estivale (8 à 33 °C) — le signal environnement→comportement y est exploitable,
contrairement à la boiterie légère (Objectif 1).

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import glob, os
from scipy.stats import spearmanr
import warnings
warnings.filterwarnings('ignore')

PROJECT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_ROOT = PROJECT / 'Données completes' / 'Données accelerometres'
REPORTS = PROJECT / 'reports' / 'objective1_pipeline_icetag'
OUT = PROJECT / 'reports' / 'objective2_environnement'
OUT.mkdir(parents=True, exist_ok=True)

ICETAG_INPUT = REPORTS / 'summer_2019_pipeline_input_15min.csv'
HOBO_DIR = DATA_ROOT / 'Summer 2019' / 'Hobo'
print('OK' if ICETAG_INPUT.exists() and HOBO_DIR.exists() else 'MANQUANT')

OK


## 1. Tâche 2.1 — Données environnementales (HOBO)

On parse et concatène les capteurs HOBO extérieurs (conditions ambiantes), résolution 5 min.

In [2]:
hobo_files = glob.glob(str(HOBO_DIR / '**' / '*.xlsx'), recursive=True)
hobo_files = [h for h in hobo_files if 'THI' not in h and ('Outside' in h or 'outside' in h)]
print(f"Fichiers HOBO extérieurs : {len(hobo_files)}")

def parse_hobo(f):
    df = pd.read_excel(f, header=1)
    datecol = [c for c in df.columns if 'Date' in str(c)][0]
    tcol = [c for c in df.columns if str(c).startswith('Temp.,')][0]
    hcol = [c for c in df.columns if str(c).startswith('HR,')]
    cols = [datecol, tcol] + hcol[:1]
    df = df[cols].copy()
    df.columns = ['dt', 'temp', 'rh'][:len(cols)]
    df['dt'] = pd.to_datetime(df['dt'], errors='coerce')
    return df.dropna(subset=['dt'])

env = pd.concat([parse_hobo(f) for f in hobo_files], ignore_index=True)
env = env.drop_duplicates(subset=['dt']).sort_values('dt')
print(f"Mesures environnementales : {len(env)}")
print(f"Période : {env['dt'].min()} -> {env['dt'].max()}")
print(f"Température : {env['temp'].min():.1f} à {env['temp'].max():.1f} °C")
print(f"Humidité : {env['rh'].min():.0f} à {env['rh'].max():.0f} %")

Fichiers HOBO extérieurs : 14


Mesures environnementales : 36392
Période : 2019-07-01 08:32:29 -> 2019-09-09 07:57:32
Température : 8.0 à 32.8 °C
Humidité : 29 à 96 %


## 2. Calcul du THI (indice de stress thermique)

Formule standard en élevage laitier :
$$THI = (1.8 \times T + 32) - (0.55 - 0.0055 \times RH) \times (1.8 \times T - 26)$$

Seuils de stress thermique : < 68 aucun, 68–71 léger, 72–79 modéré, ≥ 80 sévère.

In [3]:
def compute_thi(t, rh):
    return (1.8 * t + 32) - (0.55 - 0.0055 * rh) * (1.8 * t - 26)

env['THI'] = compute_thi(env['temp'], env['rh'])

def thi_category(thi):
    if thi < 68: return '1_aucun'
    if thi < 72: return '2_leger'
    if thi < 80: return '3_modere'
    return '4_severe'
env['THI_cat'] = env['THI'].apply(thi_category)

# Agrégation à 15 min pour matcher les bins IceTag
env = env.set_index('dt')
env15 = env[['temp', 'rh', 'THI']].resample('15min').mean().dropna()
env15['THI_cat'] = env15['THI'].apply(thi_category)
env15 = env15.reset_index()
print(f"Bins environnementaux 15 min : {len(env15)}")
print("\nRépartition des conditions thermiques :")
print(env15['THI_cat'].value_counts().sort_index().to_string())

Bins environnementaux 15 min : 6066

Répartition des conditions thermiques :
THI_cat
1_aucun     3011
2_leger     1461
3_modere    1536
4_severe      58


## 3. Tâche 2.1 — Fusion environnement × activité IceTag

L'environnement est commun au troupeau : on le joint à chaque bin de chaque vache sur le timestamp.

In [4]:
act = pd.read_csv(ICETAG_INPUT)
act['Cow'] = act['Cow'].astype(str)
act['Start'] = pd.to_datetime(act['Start'])

def hms_to_h(t):
    try:
        h, m, s = str(t).split(':'); return (int(h)*3600 + int(m)*60 + float(s)) / 3600
    except Exception:
        return np.nan
act['lying_h'] = act['Lying Time'].apply(hms_to_h)

merged = act.merge(env15, left_on='Start', right_on='dt', how='inner')
print(f"Dataset multimodal fusionné : {len(merged)} lignes ({merged['Cow'].nunique()} vaches)")
print(f"Période commune : {merged['Start'].min()} -> {merged['Start'].max()}")
merged.to_csv(OUT / 'summer2019_multimodal_15min.csv', index=False)
print('Dataset intégré sauvegardé : summer2019_multimodal_15min.csv')
merged[['Cow', 'Start', 'Steps', 'Motion Index', 'lying_h', 'temp', 'rh', 'THI', 'THI_cat']].head()

Dataset multimodal fusionné : 87501 lignes (17 vaches)
Période commune : 2019-07-01 08:30:00 -> 2019-09-06 12:00:00


Dataset intégré sauvegardé : summer2019_multimodal_15min.csv


,Cow,Start,Steps,Motion Index,lying_h,temp,rh,THI,THI_cat
0,2062,2019-07-01 08:30:00,0,0,0.250000,19.143313,76.149386,65.348639,1_aucun
1,2062,2019-07-01 08:45:00,3,21,0.147778,19.545505,75.004959,65.919259,1_aucun
2,2062,2019-07-01 09:00:00,0,0,0.250000,19.967358,75.570306,66.603498,1_aucun
3,2062,2019-07-01 09:15:00,3,15,0.118889,20.430325,74.778363,67.277292,1_aucun
4,2062,2019-07-01 09:30:00,7,7,0.006111,20.805704,73.922332,67.808376,1_aucun


## 4. Tâche 2.2 — Activité locomotrice vs température / THI

In [5]:
# Restreindre aux heures de jour (activité pertinente, 06h-20h)
merged['hour'] = merged['Start'].dt.hour
day = merged[(merged['hour'] >= 6) & (merged['hour'] < 20)].copy()

# Corrélations
print('=== Corrélations (bins de jour) ===')
for var in ['temp', 'THI', 'rh']:
    rho_s, p_s = spearmanr(day[var], day['Steps'])
    rho_m, p_m = spearmanr(day[var], day['Motion Index'])
    print(f"{var:6s} vs Pas    : rho={rho_s:+.3f} (p={p_s:.1e})  |  vs Motion Index : rho={rho_m:+.3f} (p={p_m:.1e})")

# Activité moyenne par catégorie de stress thermique
print('\n=== Activité moyenne par niveau de stress thermique (THI) ===')
byc = day.groupby('THI_cat').agg(
    n_bins=('Steps', 'size'),
    steps_moy=('Steps', 'mean'),
    mi_moy=('Motion Index', 'mean'),
    lying_h_moy=('lying_h', 'mean'),
).round(2)
print(byc.to_string())
byc.to_csv(OUT / 'summer2019_activite_par_THI.csv')

=== Corrélations (bins de jour) ===
temp   vs Pas    : rho=+0.089 (p=5.8e-91)  |  vs Motion Index : rho=+0.082 (p=1.3e-76)
THI    vs Pas    : rho=+0.097 (p=1.5e-106)  |  vs Motion Index : rho=+0.088 (p=5.7e-89)
rh     vs Pas    : rho=-0.018 (p=4.1e-05)  |  vs Motion Index : rho=-0.020 (p=4.1e-06)

=== Activité moyenne par niveau de stress thermique (THI) ===
          n_bins  steps_moy  mi_moy  lying_h_moy
THI_cat                                         
1_aucun    16586       8.61   24.44         0.11
2_leger    14169      11.64   36.49         0.10
3_modere   19574      11.21   32.93         0.10
4_severe     929      12.77   32.16         0.08


## 5. Visualisations

In [6]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

# (a) Pas moyens par tranche de température
day['temp_bin'] = pd.cut(day['temp'], bins=range(5, 36, 3))
tb = day.groupby('temp_bin')['Steps'].mean()
axes[0].bar([str(i) for i in tb.index], tb.values, color='tomato')
axes[0].set_title('Pas moyens / 15 min vs température')
axes[0].set_xlabel('Température (°C)'); axes[0].set_ylabel('Pas moyens')
axes[0].tick_params(axis='x', rotation=45)

# (b) Activité par catégorie THI
cats = byc.index.tolist()
axes[1].bar(cats, byc['steps_moy'].values, color='steelblue')
axes[1].set_title('Pas moyens par niveau de stress thermique')
axes[1].set_xlabel('Catégorie THI'); axes[1].set_ylabel('Pas moyens')
axes[1].tick_params(axis='x', rotation=45)

# (c) Profil journalier d'activité
hourly = merged.groupby('hour')['Steps'].mean()
axes[2].plot(hourly.index, hourly.values, marker='o', color='green')
axes[2].set_title("Profil journalier d'activité")
axes[2].set_xlabel("Heure"); axes[2].set_ylabel('Pas moyens')

fig.suptitle('Summer 2019 — Environnement × activité locomotrice', fontsize=13)
fig.tight_layout()
fig.savefig(OUT / 'summer2019_environnement_activite.png', dpi=120, bbox_inches='tight')
print('Figure sauvegardée :', OUT / 'summer2019_environnement_activite.png')
plt.show()

Figure sauvegardée : /Users/alioubarry/PROJECT/mcgill_iot_cattle/reports/objective2_environnement/summer2019_environnement_activite.png


## 6. Synthèse de l'Objectif 2 (Summer 2019)

In [7]:
rho_thi, p_thi = spearmanr(day['THI'], day['Steps'])
steps_aucun = byc.loc['1_aucun', 'steps_moy'] if '1_aucun' in byc.index else np.nan
steps_severe = byc.loc['4_severe', 'steps_moy'] if '4_severe' in byc.index else np.nan

lines = []
lines.append('# Objectif 2 — Environnement x comportement (Summer 2019)\n')
lines.append('## Tâche 2.1 — Synchronisation')
lines.append(f'- Dataset multimodal créé : {len(merged)} bins de 15 min, {merged["Cow"].nunique()} vaches.')
lines.append(f'- Sources fusionnées : IceTag (activité) + HOBO (température, humidité) + THI calculé.')
lines.append(f'- Période commune : {merged["Start"].min().date()} à {merged["Start"].max().date()}.\n')
lines.append('## Tâche 2.2 — Analyse')
lines.append(f'- Corrélation THI vs pas (jour) : rho = {rho_thi:+.3f} (p = {p_thi:.1e}).')
lines.append('- Activité moyenne par niveau de stress thermique :')
lines.append(byc.to_string())
lines.append('')
lines.append('## Lecture')
if not np.isnan(steps_aucun) and not np.isnan(steps_severe):
    delta = 100 * (steps_severe - steps_aucun) / steps_aucun
    lines.append(f'- En stress sévère, l\'activité varie de {delta:+.0f}% par rapport au confort thermique.')
if rho_thi < -0.05:
    lines.append('- Tendance : l\'activité DIMINUE quand le stress thermique AUGMENTE (cohérent avec le stress thermique).')
elif rho_thi > 0.05:
    lines.append('- Tendance : l\'activité augmente avec la température (cohérent avec activité accrue par temps doux/sorties).')
else:
    lines.append('- Pas de relation monotone forte THI-activité sur ce corpus.')
note = '\n'.join(lines)
(OUT / 'objectif2_summer2019_synthese.md').write_text(note, encoding='utf-8')
print(note)

# Objectif 2 — Environnement x comportement (Summer 2019)

## Tâche 2.1 — Synchronisation
- Dataset multimodal créé : 87501 bins de 15 min, 17 vaches.
- Sources fusionnées : IceTag (activité) + HOBO (température, humidité) + THI calculé.
- Période commune : 2019-07-01 à 2019-09-06.

## Tâche 2.2 — Analyse
- Corrélation THI vs pas (jour) : rho = +0.097 (p = 1.5e-106).
- Activité moyenne par niveau de stress thermique :
          n_bins  steps_moy  mi_moy  lying_h_moy
THI_cat                                         
1_aucun    16586       8.61   24.44         0.11
2_leger    14169      11.64   36.49         0.10
3_modere   19574      11.21   32.93         0.10
4_severe     929      12.77   32.16         0.08

## Lecture
- En stress sévère, l'activité varie de +48% par rapport au confort thermique.
- Tendance : l'activité augmente avec la température (cohérent avec activité accrue par temps doux/sorties).
